# Qwen3-VL VQA Prompt Test
Provide an image directory and one or more prompts. The notebook will run VQA for each image and prompt.


In [1]:
import csv
import math
import os
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm

from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info


/home/labad/miniconda3/envs/qwen3vl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# === User settings ===
IMAGE_DIR = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30"  # directory containing images (recursive)
CSV_OUTPUT_PATH = "/home/labad/minxing/code/Qwen3-VL/data_influencer22/output/test_1.csv"  # path to save results; for multiple prompts, _q1/_q2/... suffix is added

PROMPTS = [
    # People Analysis
    # "Analyze the image and determine if it contains one or more humans. Answer Yes or No. If Yes, count how many people are visible. For each visible person, provide: person_id (Person 1, Person 2, ...), recognized (true/false), name (only if explicitly recognizable, otherwise null), gender (if visually apparent, otherwise null), age_range (approximate), outfit, activity_or_pose, facial_expression, interaction_with_others (if any). Recognition rule: Only set recognized = true if the person's identity is clearly indicated in the image (e.g., name tag, text label, or well-known public figure). Do not guess or infer identity from appearance. Assign a confidence score between 0 and 1 indicating how confident you are that the labels are correct. Return valid JSON only in this format: {\"has_humans\": Boolean, \"person_count\": number, \"persons\": [ { \"person_id\": number, \"recognized\": Boolean, \"name\": string | null, \"gender\": string | null, \"age_range\": string | null, \"outfit\": string | null, \"activity_or_pose\": string | null, \"facial_expression\": string | null, \"interaction_with_others\": string | null, \"confidence\": number}]}. No additional commentary or text outside the JSON.",
    # "Analyze the image and determine if it contains a human. Answer Yes or No. If Yes, identify the most prominent person in the image (e.g., the largest, clearest, most central, or primary subject). Ignore all other people. Provide the following information for this single prominent person only: - recognized (true/false) - name (only if explicitly recognizable, otherwise null) - gender (if visually apparent, otherwise null) - age_range (approximate) - outfit - activity_or_pose - facial_expression - interaction_with_others (if any) Also assess the following based only on what is clearly visible: Presentation score (1–5): How visually polished and well-presented the person appears based on visible grooming, makeup (if any), clothing coordination, and overall styling. Scale: 1 — very unpolished. 2 — below average presentation. 3 — average, neat everyday appearance. 4 — good presentation. 5 — excellent presentation. Facial structure characteristics (set to null if the face is unclear or partially visible): - facial_geometry: Low / Medium / High - facial_symmetry: Low / Medium / High - facial_proportions: Low / Medium / High Return valid JSON only in the following format: {   \"has_human\": Boolean,  \"person\": {    \"recognized\": boolean,    \"name\": string | null,    \"gender\": string | null,    \"age_range\": string | null,    \"outfit\": string | null,    \"activity_or_pose\": string | null,    \"facial_expression\": string | null,    \"interaction_with_others\": string | null,    \"presentation_score\": number | null,    \"facial_geometry\": \"Low\" | \"Medium\" | \"High\" | null,    \"facial_symmetry\": \"Low\" | \"Medium\" | \"High\" | null,    \"facial_proportions\": \"Low\" |\"Medium\" | \"High\" | null } } If no human is present, set \"has_human\" to false and \"person\" to null. Do not include any text outside the JSON object.",
    # "Analyze the image and determine if it contains a human. Answer Yes or No. If Yes, identify the most prominent person in the image (e.g., the largest, clearest, most central, or primary subject). Ignore all other people. Provide the following information for this single prominent person only: - recognized (true/false) - name (only if explicitly recognizable, otherwise null) - gender (if visually apparent, otherwise null) - age_range (approximate) - outfit - activity_or_pose - facial_expression - interaction_with_others (if any) Also assess the following based only on what is clearly visible: Presentation score (1–5): How visually polished and well-presented the person appears based on visible grooming, makeup (if any), clothing coordination, and overall styling. Scale: 1 — very unpolished. 2 — below average presentation. 3 — average, neat everyday appearance. 4 — good presentation. 5 — excellent presentation. Facial structure characteristics (set to null if the face is unclear or partially visible):  facial_geometry: overall facial structure and alignment (Low / Medium / High). - facial_symmetry: degree of visible left-right facial balance (Low / Medium / High). - facial_proportions: how balanced the facial proportions appear relative to common facial ratios (Low / Medium / High). Return valid JSON only in the following format: {   \"has_human\": Boolean,  \"person\": {    \"recognized\": boolean,    \"name\": string | null,    \"gender\": string | null,    \"age_range\": string | null,    \"outfit\": string | null,    \"activity_or_pose\": string | null,    \"facial_expression\": string | null,    \"interaction_with_others\": string | null,    \"presentation_score\": number | null,    \"facial_geometry\": \"Low\" | \"Medium\" | \"High\" | null,    \"facial_symmetry\": \"Low\" | \"Medium\" | \"High\" | null,    \"facial_proportions\": \"Low\" |\"Medium\" | \"High\" | null } } If no human is present, set \"has_human\" to false and \"person\" to null. Do not include any text outside the JSON object.",
    # "Analyze the image and determine if it contains a human. Answer Yes or No. If Yes, identify the most prominent person in the image (e.g., the largest, clearest, most central, or primary subject). Ignore all other people. Provide the following information for this single prominent person only: - recognized (true/false) - name (only if explicitly recognizable, otherwise null) - gender (if visually apparent, otherwise null) - age_range (approximate) - outfit - activity_or_pose - facial_expression - interaction_with_others (if any) Also assess the following based only on what is clearly visible Attractiveness score (1-5) (set to null if the face is unclear or partially visible): Based on Western standards of attractiveness, and general agreement among most people, how attractive is the person in the image? Please provide a rating on a scale from 1 to 5. For example 1 – Very unattractive 2 – Unattractive 3 – Average 4 – Attractive 5 – Very attractive Presentation score (1–5): How visually polished and well-presented the person appears based on visible grooming, makeup (if any), clothing coordination, and overall styling. Scale: 1 — very unpolished. 2 — below average presentation. 3 — average, neat everyday appearance. 4 — good presentation. 5 — excellent presentation. Facial structure characteristics (set to null if the face is unclear or partially visible): - facial_geometry: Low / Medium / High - facial_symmetry: Low / Medium / High - facial_proportions: Low / Medium / High Return valid JSON only in the following format: { \"has_human\": boolean, \"person\": { \"recognized\": boolean, \"name\": string | null, \"gender\": string | null, \"age_range\": string | null, \"outfit\": string | null, \"activity_or_pose\": string | null, \"facial_expression\": string | null, \"interaction_with_others\": string | null, \"attractive\": number, \"presentation_score\": number | null, \"facial_geometry\": \"Low\" | \"Medium\" | \"High\" | null, \"facial_symmetry\": \"Low\" | \"Medium\" | \"High\" | null, \"facial_proportions\": \"Low\" | \"Medium\" | \"High\" | null } } If no human is present, set \"has_human\" to false and \"person\" to null. Do not include any text outside the JSON object.",
    # "Analyze the image and determine if it contains one or more humans. Answer Yes or No. If Yes, count how many people are visible. Identify the most prominent person in the image (e.g., the largest, clearest, most central, or primary subject). Ignore all other people. Provide the following information for this single prominent person only: - recognized (true/false) - name (only if explicitly recognizable, otherwise null) - gender (if visually apparent, otherwise null) - age_range (approximate, only in numbers) - outfit - activity_or_pose - facial_expression - interaction_with_others (if any)  Assign a confidence score between 0 and 1, indicating how confident you are that the prominent person is correct. Also, assess the following based only on what is clearly visible: Attractiveness score (1-5) (set to null if the face is unclear or partially visible): Based on Western standards of attractiveness, and general agreement among most people, how attractive is the person in the image? Please provide a rating on a scale from 1 to 5. For example 1 – Very unattractive 2 – Unattractive 3 – Average 4 – Attractive 5 – Very attractive. Provide a one-sentence justification for the attractiveness score. Presentation score (1–5): How visually polished and well-presented the person appears based on visible grooming, makeup (if any), clothing coordination, and overall styling. Scale: 1 — very unpolished. 2 — below average presentation. 3 — average, neat everyday appearance. 4 — good presentation. 5 — excellent presentation.  Facial structure characteristics (set to null if the face is unclear or partially visible): - facial_geometry: Low / Medium / High - facial_symmetry: Low / Medium / High - facial_proportions: Low / Medium / High Return valid JSON only in the following format: { \"has_human\": Boolean,  \"person_count\": number, \"person\": { \"recognized\": boolean, \"name\": string | null, \"gender\": string | null, \"age_range\": string | null, \"outfit\": string | null, \"activity_or_pose\": string | null, \"facial_expression\": string | null, \"interaction_with_others\": string | null, \"attractive\": number | null, \"attractiveness_justification\": string | null, \"presentation_score\": number | null, \"facial_geometry\": \"Low\" | \"Medium\" | \"High\" | null, \"facial_symmetry\": \"Low\" | \"Medium\" | \"High\" | null, \"facial_proportions\": \"Low\" | \"Medium\" | \"High\" | null, \"confidence\": number}|null}",
    # Type & Quality & Aeththetics
    # "Classify the image format into exactly one label using the definitions below. Return the label name only (not the description). Formats (label: description): photo — real-world camera photo of people, places, or objects (not a UI capture). screenshot — capture of an app or website showing UI elements (menus, buttons, bars). news_card — designed news graphic, broadcast overlay, weather alert, or security footage presenting a headline (not a screenshot). flyer_poster — event or announcement layout with date, time, location, or call-to-action. advertisement_graphic — promotional graphic for a product, brand, or service. infographic — visual explanation using charts, statistics, or structured facts. document — scanned or photographed document, form, or letter. painting — hand-drawn, painted, sketched, illustrated, or comic-style artwork. meme — humorous or reaction-style image using a meme format. other — none of the above formats clearly apply. If multiple formats seem possible, choose the closest single match. Use \"other\" only if no format reasonably fits. Also provide two 1–5 ratings:  Aesthetic score (visual appeal/composition, independent of resolution): 1 — very poor: cluttered, awkward layout, unpleasant visual balance. 2 — below average: weak composition or distracting arrangement. 3 — average: acceptable, clear, but not visually strong. 4 — good: well-composed, balanced, visually pleasing. 5 — excellent: highly appealing, polished, professional-looking. Quality score (technical image quality: sharpness, resolution, noise, artifacts): 1 — very poor: blurry/low-res, heavy artifacts, hard to view. 2 — below average: noticeable blur/low-res or artifacts. 3 — average: usable and readable, minor flaws. 4 — good: sharp, clear, minimal artifacts. 5 — excellent: very sharp, high-res, clean and well-exposed. Assign a confidence score between 0 and 1 indicating how confident you are that the image_type classification is correct. Return valid JSON only: {  \"image_type\": \"<one label>\",  \"aesthetic_score\": number,  \"quality_score\": number,  \"confidence\": number } Do not return multiple labels or include any text outside the JSON.",
    # "Classify the image format into exactly one label using the definitions below. Return the label name only (not the description). Formats (label: description): photo — real-world camera photo of people, places, or objects (not a UI capture). screenshot — capture of an app or website showing UI elements (menus, buttons, bars). news_card — designed news graphic, broadcast overlay, weather alert, or security footage presenting a headline (not a screenshot). flyer_poster — event or announcement layout with date, time, location, or call-to-action. advertisement_graphic — promotional graphic for a product, brand, or service. infographic — visual explanation using charts, statistics, or structured facts. document — scanned or photographed document, form, or letter. painting — hand-drawn, painted, sketched, illustrated, or comic-style artwork. meme — humorous or reaction-style image using a meme format. other — none of the above formats clearly apply. If multiple formats seem possible, choose the closest single match. Use \"other\" only if no format reasonably fits. Also provide two 1–5 ratings:  Aesthetic score (visual appeal/composition, independent of resolution): 1 — very poor: cluttered, awkward layout, unpleasant visual balance. 2 — below average: weak composition or distracting arrangement. 3 — average: acceptable, clear, but not visually strong. 4 — good: well-composed, balanced, visually pleasing. 5 — excellent: highly appealing, polished, professional-looking. Quality score (technical image quality: sharpness, resolution, noise, artifacts): 1 — very poor: blurry/low-res, heavy artifacts, hard to view. 2 — below average: noticeable blur/low-res or artifacts. 3 — average: usable and readable, minor flaws. 4 — good: sharp, clear, minimal artifacts. 5 — excellent: very sharp, high-res, clean and well-exposed. Assign a confidence score between 0 and 1 indicating how confident you are that the image_type classification is correct and a one-sentence justification. Return valid JSON only: {  \"image_type\": \"<one label>\",  \"aesthetic_score\": number,  \"quality_score\": number,  \"confidence\": number, \"justification\": string } Do not return multiple labels or include any text outside the JSON.",
    # OCR
    # "You are an OCR and layout extraction model. Process the image in two distinct phases: Phase 1: Raw Transcription Detect and transcribe: EVERY piece of visible text in the image exactly as it appears. Include everything: headlines, body text, logos, brand names, and watermarks. Do not omit anything. Preserve capitalization and line breaks. Phase 2: Extract the text based on these rules: 1. Headline: The most prominent large text. 2. Subtitle: The smaller supporting text directly associated with the headline. 3. Exclusion Rule: In this phase ONLY, discard all brand names, logos (e.g., \"12NEWS\"), URLs, and watermarks. Output Format: Output the result as valid JSON only: { \"has_text\": boolean, \"transcription\": \"String containing EVERY word from Phase 1\", \"headline\": \"String or null\", \"subtitle\": \"String or null\", \"confidence\": float }",
    # "You are an OCR and layout extraction model. Process the image in two phases. Phase 1: Raw Transcription — Detect and transcribe EVERY piece of visible text in the image exactly as it appears. Include all readable text including headlines, body text, brand names, logos that contain readable text, watermarks, captions, and labels. Do not omit, summarize, infer, or correct anything. Preserve original capitalization and reading order. Preserve line breaks using \n escape characters so the output remains valid JSON. Transcribe text in any language exactly as shown. Phase 2: Structured Metadata Extraction — Using only the transcription from Phase 1, extract structured metadata. Headline: the most prominent non-brand text in the image (typically the largest font); if the most prominent text is a brand name or logo, return null. Subtitle: smaller supporting text clearly associated with the headline; may include multiple lines; if no clearly associated subtitle exists, return null. Exclusion Rule (Phase 2 ONLY): discard all brand names, logos, URLs, and watermarks from headline and subtitle. Output Requirements: output valid JSON only with no explanations, markdown, or extra text, using this schema exactly: { \"has_text\": boolean, \"transcription\": \"String containing EVERY word from Phase 1 with line breaks as \\n\", \"headline\": \"String or null\", \"subtitle\": \"String or null\", \"confidence\": float }. Field Definitions: has_text is true if at least one visible alphanumeric character is detected; otherwise false. transcription is the complete raw transcription from Phase 1. headline is the extracted headline or null. subtitle is the extracted subtitle or null. confidence is a float between 0.0 and 1.0 indicating how confident the model is that the OCR transcription is correct. Strict Rules: do not hallucinate missing text, do not merge Phase 1 and Phase 2, do not output invalid JSON, and if no text exists return empty strings for text fields and null for headline and subtitle.",
    # Topic
    # "Choose the primary topic of the image. Pick exactly ONE topic_id from the list below and return the topic_id only (not the description text). Use \"other\" ONLY if none of the listed topics clearly apply. Topics (topic_id: description): politics: government, elections, public policy, political actors. crime: criminal acts, violence, investigations, arrests. natural_disaster: earthquakes, floods, fires, storms, natural damage. entertainment: celebrities, movies, music, TV, pop culture. sports: athletic events, teams, players, competitions. technology: software, hardware, AI, digital products. science: research, discoveries, space, experiments. health: medicine, disease, public health, wellness. fitness: exercise, workouts, physical training. gaming: video games, esports, gaming culture. anime: anime or manga-related content. fashion: clothing, style, accessories. beauty_makeup: cosmetics, skincare, makeup, appearance. food: cooking, meals, restaurants. pets_animals: animals or pets as the main subject. nature: landscapes, wildlife, natural scenes. travel: tourism, destinations, travel experiences. art: creative artwork or artistic expression. photography: photography as a subject (gear, technique, portfolios). religion: religious beliefs, rituals, symbols. history: historical events figures or eras memes humorous or reaction-style meme content advertising promotional or marketing-focused content daily_life everyday personal moments or routine activities other none of the above topics clearly apply If multiple topics seem possible choose the closest single match Only use \"other\" if no category reasonably fits Also output a confidence score from 0 to 1 and a one-sentence justification Return JSON only {  \"topic\": \"<topic_id>\",  \"confidence\": number  \"justification\": string} Do not include any text outside the JSON.",
    # Image Description
    # "Describe the image using short, factual phrases. Only describe what is clearly visible and avoid assumptions. Fields: - people: visible people or their appearance. - objects: key visible objects. - action: what is happening. - background: setting or environment. - visible_text: any clearly readable text. Use at most 12 words per field. If a field is not applicable, set it to null. Assign a confidence score between 0 and 1 indicating how confident you are that this description accurately represents the image. Use high confidence only if the scene is clear and unambiguous. Return valid JSON only: { \"people\": string | null, \"objects\": string | null, \"action\": string | null, \"background\": string | null, \"visible_text\": string | null, \"confidence\": number } Do not include any text outside the JSON object.",
    # Safety & Toxicity
    # "You are an image safety and content analysis model. Analyze the provided image to determine whether it contains any toxic, harmful, unsafe, or sensitive content. This task is for dataset analysis and labeling, not moderation or blocking. Evaluate the image using the following categories based strictly on visible content: 1. Hate, Humiliation, or Harassment (e.g., hate symbols, slurs, degrading or threatening behavior toward individuals or groups) 2. Violence, Harm, or Cruelty (e.g., physical assault, weapons in use, bloodshed, torture, severe injury) 3. Sexual Content (e.g., sexual acts or explicit sexualized behavior) 4. Nudity Content (e.g., exposed genitals or breasts; exclude clearly neutral medical or artistic contexts) 5. Criminal Planning or Activity (e.g., theft, assault, terrorism, or crime preparation) 6. Illegal Weapons or Substance Abuse (e.g., illegal drugs, drug use, unlicensed firearms, bomb-making materials) 7. Self-Harm or Suicide (e.g., self-injury, suicide attempts, encouragement of self-harm) 8. Animal Cruelty (e.g., abuse, torture, or killing of animals outside lawful hunting or food preparation) 9. Disasters or Emergencies (e.g., fires, floods, explosions, accidents, or medical emergencies) Rules:- Base classification only on what is visually observable. - Do not infer intent or context beyond the image. - Do not describe graphic details. - Multiple categories may be true for a single image. Primary Category: - Select one primary_category representing the most dominant or central form of toxicity present. - If no toxic content is detected, set primary_category to null. Human Review Requirement: - Set human_review_required to true ONLY if the model is uncertain due to ambiguous, unclear, borderline, or conflicting visual evidence. - Do not use fixed thresholds to determine this; rely on overall uncertainty. Output valid JSON only using the schema below, with no explanations or extra text: {  \"is_toxic\": Boolean, \"primary_category\": \"string or null\", \"categories\": {    \"hate_harassment\": boolean,    \"violence_harm\": boolean,    \"sexual_content\": Boolean,    \"nudity\": Boolean,    \"criminal_activity\": boolean,    \"illegal_weapons_or_drugs\": Boolean,    \"self_harm\": Boolean,    \"animal_cruelty\": boolean,    \"disaster_or_emergency\": Boolean  },  \"confidence\": float,  \"human_review_required\": Boolean} Definitions: - is_toxic is true if any category is true. - confidence is a float between 0.0 and 1.0 representing how confident the model is in its classification.",
    # Toxicity 1
    # "You are an image safety and content analysis model. Analyze the provided image and determine whether it contains any toxic, harmful, unsafe, or sensitive content based strictly on visible elements in the image (objects, actions, symbols, and text). Evaluate the image using the following categories: 1. Hate, Humiliation, or Harassment (e.g., hate symbols, slurs, degrading or threatening behavior toward individuals or groups) 2. Violence, Harm, or Cruelty (e.g., physical assault, weapons in use, bloodshed, torture, severe injury) 3. Sexual Content (e.g., sexual acts or explicit sexualized behavior) 4. Nudity Content (e.g., exposed genitals or breasts; exclude clearly neutral medical or artistic contexts) 5. Criminal Planning or Activity (e.g., theft, assault, terrorism, or crime preparation) 6. Illegal Weapons or Substance Abuse (e.g., illegal drugs, visible drug use, bomb-making materials; do not assume illegality unless visually obvious) 7. Self-Harm or Suicide (e.g., self-injury, suicide attempts, encouragement of self-harm) 8. Animal Cruelty (e.g., abuse, torture, or killing of animals outside lawful hunting or food preparation) 9. Disasters or Emergencies (e.g., fires, floods, explosions, accidents, or medical emergencies) Rules:- Base classification only on visible content; do not infer intent beyond what is visually evident.- Do not describe graphic details.- Multiple categories may be true for a single image.- Fictional, cartoon, or animated depictions of harmful content must still be classified; adjust scale based on severity and realism. Primary Category:- Select one primary_category representing the most severe or dominant form of toxicity present.- If no toxic content is detected, set primary_category to null. Output valid JSON only using the schema below, with no explanations or extra text:{\"is_toxic\": Boolean,\"primary_category\": \"string or null\",\"categories\": {\"hate_harassment\": boolean, \"violence_harm\": boolean, \"sexual_content\": Boolean, \"nudity\": Boolean, \"criminal_activity\": boolean, \"illegal_weapons_or_drugs\": Boolean, \"self_harm\": Boolean, \"animal_cruelty\": boolean, \"disaster_or_emergency\": Boolean}, \"confidence\": float, \"scale\": number, \"human_review_required\": Boolean} Definitions:- is_toxic is true if any category is true.- confidence is a float between 0.0 and 1.0 representing certainty based on visual clarity and category distinctness.- scale is an integer from 1 to 5 indicating toxicity severity. Scale Definition (1–5): 1 = Least toxic: very mild or borderline harmful content (e.g., symbolic or cartoonish harm, mild non-graphic references) 2 = Low toxicity: mild harmful content with limited impact (e.g., non-graphic low-intensity violence, mild harassment) 3 = Moderate toxicity: clearly harmful content without extreme explicitness (e.g., visible violence, hate symbols, implied self-harm) 4 = High toxicity: explicit or intense harmful content (e.g., serious injury, explicit sexual content or nudity, clear criminal acts) 5 = Highly toxic: extreme, graphic, or severely disturbing content (e.g., graphic violence, torture, suicide attempts, severe sexual violence)",
    # Toxicity 2
    # "You are an image safety and content analysis model. Analyze the provided image and determine whether it contains any toxic, harmful, unsafe, or sensitive content based strictly on visible elements in the image (objects, actions, symbols, and text). Do consider fictional, cartoon, or animated images if they depict harmful content. Provide multiple labels if required. Do not describe graphic details. Base classification only on visible content; do not infer intent beyond what is visually evident. Provide a confidence (0.0–1.0) indicating how confident you are. Also scale from 1 to 5 indicating toxicity severity where 1 is least toxic and 5 is highly toxic. Give a one-sentence justification. Output valid JSON only using the schema below, with no explanations or extra text:{\"is_toxic\":Boolean,\"labels\": [\"string\"] or null,\"confidence\": float,\"scale\": number, \"justification\": string}",


    # Influencer_22 specific prompts
    # 
    "You are helping with a research annotation task on Instagram influencer images. Task: Look at the image and identify the PRIMARY communicative intent of the post image. Important rules: - Focus mainly on the visual content of the image. - Do not infer hidden intentions unless there is strong visual evidence. - Choose the single best primary intent. - If multiple intents seem possible, choose the dominant one and mention the secondary one in the explanation. - Be conservative. Do not over-interpret. Intent categories: 1. self_presentation. Definition: The image mainly presents the person's appearance, identity, persona, status, attractiveness, or personal brand. Examples: posed portraits, glamour shots, red carpet looks, image-centered solo photos. 2. lifestyle_display. Definition: The image mainly showcases a desirable lifestyle, setting, experience, routine, consumption pattern, or social world. Examples: travel, luxury settings, workouts, dining, parties, vacations, daily life aesthetics. 3. intimacy_building. Definition: The image mainly creates a sense of personal closeness, vulnerability, warmth, behind-the-scenes access, or \"sharing life with followers\". Examples: casual home moments, family scenes, candid emotional moments, low-polish personal snapshots. 4. promotion. Definition: The image mainly promotes a product, brand, event, collaboration, campaign, or commercial identity. Examples: holding products prominently, campaign-style branded images, event posters, clear sponsorship aesthetics. 5. information_sharing. Definition: The image mainly conveys information, news, updates, instructions, announcements, or public communication. Examples: infographic-like posts, event notices, update posts, awareness or informational slides. 6. entertainment. Definition: The image mainly aims to amuse, entertain, surprise, or perform spectacle. Examples: funny visuals, memes, playful performance, exaggerated staged humor. 7. activism. Definition: The image mainly expresses support for a cause, movement, social issue, political stance, or public advocacy. Examples: protest-related imagery, awareness visuals, symbolic support for a cause. Output format: Return valid JSON only, with exactly these fields: { \"primary_intent\": \"...\", \"secondary_intent\": \"... or null\", \"confidence\": 0.00, \"reasoning\": \"...\", \"visual_evidence\": [\"...\", \"...\", \"...\"] } Constraints: - primary_intent must be one of [\"self_presentation\",\"lifestyle_display\",\"intimacy_building\",\"promotion\",\"information_sharing\",\"entertainment\",\"activism\"] - secondary_intent must be one of the same labels or null - confidence must be between 0 and 1 - reasoning should be 1-3 sentences - visual_evidence should list concrete visible cues only - Do not mention anything not visually inferable unless clearly visible in the image",
    # visual enhancement
    "You are helping with a research annotation task on Instagram influencer images. Task: Evaluate whether this image shows obvious signs of visible visual enhancement. Important rules: - Use only visible evidence from the image. - Do not use outside knowledge about the person, brand, or event. - Judge visual appearance only, not the actual editing process. - Be conservative. If uncertain, give a middle score. Definition: Visible visual enhancement refers to any noticeable visual modification or beautification that makes the image look intentionally enhanced, decorated, stylized, or edited beyond a plain natural capture. This may include, but is not limited to: - beauty filter effects - skin smoothing - glam retouching - facial or body beautification - decorative stickers or overlays - AR effects - unusually stylized color grading - highly edited visual polish. Scoring: 1 = no clear visible enhancement. 2 = slight enhancement. 3 = moderate / uncertain. 4 = strong enhancement. 5 = very strong and obvious enhancement. Return valid JSON only: {\"visible_visual_enhancement\": 0, \"confidence\": 0.0, \"reasoning\": \"...\", \"visual_evidence\": [\"...\", \"...\", \"...\"]} Constraints: - visible_visual_enhancement must be an integer from 1 to 5. - confidence must be between 0 and 1. - reasoning should be 1-2 sentences. - visual_evidence must contain only concrete visible cues.",
    # gendered_presentation_cues
    "You are helping with a research annotation task on Instagram influencer images. Task: Evaluate whether this image contains obvious gendered presentation cues. Important rules: - Use only visible evidence from the image. - Do not infer the person's actual gender identity. - Focus on presentation style, not identity. - Be conservative. If uncertain, give a middle score. Definition: Gendered presentation cues refer to visible appearance, styling, pose, body display, grooming, makeup, clothing, or visual emphasis that strongly draws on culturally recognizable femininity-coded or masculinity-coded presentation norms. Possible examples include: - glam makeup - emphasized lashes or lips - highly feminized beauty styling - muscular torso emphasis - strong masculinity-coded pose or styling - sexualized femininity - athleticized masculinity - strongly gender-coded fashion or grooming. Scoring: 1 = no clear gendered presentation cues. 2 = slight cues. 3 = moderate / uncertain. 4 = strong cues. 5 = very strong and obvious cues. Return valid JSON only: {\"gendered_presentation_cues\": 0, \"confidence\": 0.0, \"reasoning\": \"...\", \"visual_evidence\": [\"...\", \"...\", \"...\"]} Constraints: - gendered_presentation_cues must be an integer from 1 to 5. - confidence must be between 0 and 1. - reasoning should be 1-2 sentences. - visual_evidence must contain only concrete visible cues. - Do not label the person’s identity; evaluate only visible presentation.",
    # ai_like_synthetic_feel
    "You are helping with a research annotation task on Instagram influencer images. Task: Evaluate whether this image visually feels AI-generated, synthetic, virtual, or uncanny. Important rules: - Use only visible evidence from the image. - Do not judge whether the image was actually AI-generated. - Judge only whether it LOOKS synthetic, uncanny, virtual, or AI-like. - Do not use outside knowledge about the person. - Be conservative. If uncertain, give a middle score. Definition: AI-like synthetic feel refers to a visual impression that the image appears unnaturally artificial, virtual, uncanny, overly perfect, or machine-generated in style, regardless of whether it was actually created by AI. Possible cues include: - unnatural textures - uncanny facial details - unrealistic anatomy - oddly perfect surfaces - hyper-clean synthetic look - virtual-influencer-like appearance - strange lighting or rendering artifacts. Scoring: 1 = clearly natural-looking. 2 = slightly synthetic-looking. 3 = mixed / uncertain. 4 = strongly synthetic-looking. 5 = very strongly uncanny / virtual / AI-like. Return valid JSON only: {\"ai_like_synthetic_feel\": 0, \"confidence\": 0.0, \"reasoning\": \"...\", \"visual_evidence\": [\"...\", \"...\", \"...\"]} Constraints: - ai_like_synthetic_feel must be an integer from 1 to 5. - confidence must be between 0 and 1. - reasoning should be 1-2 sentences. - visual_evidence must contain only concrete visible cues.",
]
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
BATCH_SIZE = 4
NUM_SAMPLES = None  # None = all images

# Optional sampling params
MAX_NEW_TOKENS = 500 # 200 default
TEMPERATURE = 0.6
TOP_P = 0.9
TOP_K = 50
REPETITION_PENALTY = 1.05

# Memory/precision knobs
DTYPE = "bfloat16"  # auto|bfloat16|float16|float32
FLASH_ATTN2 = False
DEVICE_MAP = "auto"

MIN_PIXELS = None  # e.g., 256 * 32 * 32
MAX_PIXELS = None  # e.g., 384 * 32 * 32

# Confidence uses token logprobs from generation.
OUTPUT_CONFIDENCE = True  # print (conf=...) at end of each answer


In [3]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".heic", ".heif"}

def find_images(root: str):
    paths = []
    for r, _, files in os.walk(root):
        for f in files:
            if Path(f).suffix.lower() in IMG_EXTS:
                paths.append(os.path.join(r, f))
    paths.sort()
    return paths

def load_images(paths):
    imgs, valid_paths = [], []
    for p in paths:
        try:
            imgs.append(Image.open(p).convert("RGB"))
            valid_paths.append(p)
        except Exception as e:
            print(f"[SKIP] {p}: {e}")
    return imgs, valid_paths


In [4]:
dtype = "auto"
if DTYPE.lower() in ("bfloat16", "bf16"):
    dtype = torch.bfloat16
elif DTYPE.lower() in ("float16", "fp16", "half"):
    dtype = torch.float16
elif DTYPE.lower() in ("float32", "fp32"):
    dtype = torch.float32

attn_impl = "flash_attention_2" if FLASH_ATTN2 else "sdpa"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "left"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    attn_implementation=attn_impl,
    device_map=DEVICE_MAP,
)
model.eval()
device = model.device
print(f"Loaded model on {device}; dtype={model.dtype}; attn={attn_impl}")

gen_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    repetition_penalty=REPETITION_PENALTY,
)
if OUTPUT_CONFIDENCE:
    gen_kwargs.update(dict(return_dict_in_generate=True, output_scores=True))


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.48it/s]


Loaded model on cuda:0; dtype=torch.bfloat16; attn=sdpa


In [5]:
def compute_confidence(scores, sequences, input_ids, tokenizer):
    if scores is None:
        return [None] * sequences.size(0)

    gen_ids = sequences[:, input_ids.shape[1]:]
    eos_id = getattr(tokenizer, "eos_token_id", None)
    confs = []

    for i in range(gen_ids.size(0)):
        tokens = gen_ids[i].tolist()
        length = len(tokens)
        if eos_id is not None and eos_id in tokens:
            length = tokens.index(eos_id)
        if length == 0:
            confs.append(None)
            continue

        logprob_sum = 0.0
        count = 0
        steps = min(length, len(scores))
        for t in range(steps):
            logits = scores[t][i]
            logprob = torch.log_softmax(logits, dim=-1)
            logprob_sum += logprob[tokens[t]].item()
            count += 1

        if count == 0:
            confs.append(None)
        else:
            confs.append(float(math.exp(logprob_sum / count)))

    return confs


def _csv_path_for_question(base_path, q_idx, total_questions):
    """Return base_path unchanged for a single prompt; add _q{idx} suffix for multiple."""
    if total_questions == 1:
        return base_path
    p = Path(base_path)
    return str(p.parent / f"{p.stem}_q{q_idx}{p.suffix}")


image_paths = find_images(IMAGE_DIR)
if NUM_SAMPLES is not None:
    image_paths = image_paths[:NUM_SAMPLES]
print(f"Found {len(image_paths)} images")

for q_idx, question in enumerate(PROMPTS, start=1):
    print(f"\nQuestion {q_idx}: {question}")

    out_path = _csv_path_for_question(CSV_OUTPUT_PATH, q_idx, len(PROMPTS))
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    csv_file = open(out_path, "w", newline="", encoding="utf-8")
    csv_writer = csv.DictWriter(csv_file, fieldnames=["filename", "answer"])
    csv_writer.writeheader()

    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc=f"Q{q_idx}"):
        batch_paths = image_paths[i : i + BATCH_SIZE]
        batch_imgs, batch_paths = load_images(batch_paths)

        if not batch_imgs:
            continue

        messages = []
        for img in batch_imgs:
            image_content = {"type": "image", "image": img}
            if MIN_PIXELS is not None:
                image_content["min_pixels"] = MIN_PIXELS
            if MAX_PIXELS is not None:
                image_content["max_pixels"] = MAX_PIXELS

            messages.append([
                {
                    "role": "user",
                    "content": [
                        image_content,
                        {"type": "text", "text": question},
                    ],
                }
            ])

        texts = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        images, videos, video_kwargs = process_vision_info(
            messages,
            image_patch_size=16,
            return_video_kwargs=True,
            return_video_metadata=True,
        )

        if videos is not None:
            videos, video_metadatas = zip(*videos)
            videos, video_metadatas = list(videos), list(video_metadatas)
        else:
            video_metadatas = None

        inputs = processor(
            text=texts,
            images=images,
            videos=videos,
            video_metadata=video_metadatas,
            padding=True,
            return_tensors="pt",
            do_resize=False,
            **video_kwargs,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        if OUTPUT_CONFIDENCE:
            sequences = outputs.sequences
            scores = outputs.scores
        else:
            sequences = outputs
            scores = None

        trimmed = [out[len(inp) :] for inp, out in zip(inputs.input_ids, sequences)]
        answers = processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

        confidences = compute_confidence(
            scores, sequences, inputs.input_ids, processor.tokenizer
        ) if OUTPUT_CONFIDENCE else [None] * len(answers)

        for p, ans, conf in zip(batch_paths, answers, confidences):
            ans = (ans or "").strip().replace("\n", " ")
            if conf is None:
                print(f"{p}: {ans}")
            else:
                print(f"{p}: {ans} (conf={conf:.4f})")
            csv_writer.writerow({"filename": p, "answer": ans})

    csv_file.close()
    print(f"Saved rows -> {out_path}")


Found 30 images

Question 1: You are helping with a research annotation task on Instagram influencer images. Task: Look at the image and identify the PRIMARY communicative intent of the post image. Important rules: - Focus mainly on the visual content of the image. - Do not infer hidden intentions unless there is strong visual evidence. - Choose the single best primary intent. - If multiple intents seem possible, choose the dominant one and mention the secondary one in the explanation. - Be conservative. Do not over-interpret. Intent categories: 1. self_presentation. Definition: The image mainly presents the person's appearance, identity, persona, status, attractiveness, or personal brand. Examples: posed portraits, glamour shots, red carpet looks, image-centered solo photos. 2. lifestyle_display. Definition: The image mainly showcases a desirable lifestyle, setting, experience, routine, consumption pattern, or social world. Examples: travel, luxury settings, workouts, dining, parties,

Q1:  12%|█▎        | 1/8 [02:05<14:36, 125.22s/it]

/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/10.jpg: {   "primary_intent": "self_presentation",   "secondary_intent": "lifestyle_display",   "confidence": 0.85,   "reasoning": "The image centers on the individual (West McKennie) as the main subject, showcasing his appearance, style, and relaxed demeanor through deliberate posing. While the van setting hints at a lifestyle, the focus remains on presenting his persona.",   "visual_evidence": [     "West McKennie is prominently featured in the foreground",     "He is wearing stylish clothing (patterned shirt, sunglasses, tattoos visible)",     "His pose and expression are intentional and composed",     "The Instagram handle 'west.mckennie' confirms the post is personal branding"   ] } (conf=0.8429)
/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/101.jpg: {   "primary_intent": "information_sharing",   "secondary_intent": "self_presentation",   "confidence": 0.95,   "reasoning": "The dominant visual element is

Q1:  25%|██▌       | 2/8 [04:37<14:06, 141.14s/it]

/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/124.webp: {   "primary_intent": "self_presentation",   "secondary_intent": null,   "confidence": 0.98,   "reasoning": "The image features Mohamed Salah in formal attire, smiling while holding an award, which centers on presenting his identity, status, and achievements as a celebrated footballer. The pose and direct gaze reinforce personal branding.",   "visual_evidence": [     "Mohamed Salah in a suit and tie",     "Smiling directly at camera",     "Holding a trophy with his name engraved",     "Formal studio background"   ] } (conf=0.9019)
/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/135.jpg: {   "primary_intent": "self_presentation",   "secondary_intent": null,   "confidence": 0.95,   "reasoning": "The image centers on the individual’s appearance and identity as a professional athlete, highlighted by his focused expression and team uniform. His gesture (blowing a kiss) adds a personal, persona-enhancing el

...

/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/371.jpg: {   "primary_intent": "information_sharing",   "secondary_intent": null,   "confidence": 0.98,   "reasoning": "The image prominently displays a match result ('FULL TIME 2-2') with team logos and player action, clearly communicating sports event information. The framing and text layout prioritize conveying facts over personal expression or lifestyle.",   "visual_evidence": [     "Text 'FULL TIME 2-2' prominently displayed at top",     "Barcelona and Juventus club logos shown with score",     "Player photo framed as part of a results graphic",     "Logos 'DALLAS' and 'JUVE ON THE ROAD' indicating event context"   ] } (conf=0.8951)
/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/380.jpg: {   "primary_intent": "information_sharing",   "secondary_intent": null,   "confidence": 0.98,   "reasoning": "The image prominently displays text announcing 'WINNERS FA YOUTH CUP CLASS OF 22' and features a collage of pl

Q1:  88%|████████▊ | 7/8 [14:49<02:04, 124.67s/it]

/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/440.jpg: {   "primary_intent": "entertainment",   "secondary_intent": null,   "confidence": 0.98,   "reasoning": "The image uses a meme format with text overlays that parody modern gaming culture applied to chess, creating humorous exaggeration. The boy’s frustrated pose reinforces the comedic intent. There is no visual evidence of self-presentation, lifestyle display, or other intents.",   "visual_evidence": [     "Text overlay: 'If Chess was invented today'",     "Text overlay: 'This map sucks'",     "Text overlay: 'Why is queen so op'",     "Text overlay: 'This game mode is stupid'",     "Boy covering face in frustration"   ] } (conf=0.9416)
/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/469.webp: {   "primary_intent": "promotion",   "secondary_intent": null,   "confidence": 0.98,   "reasoning": "The image prominently features branding for 'Fantasy Football' and 'PS5', along with a call to action 'PLAY NOW 

Q1: 100%|██████████| 8/8 [15:23<00:00, 115.45s/it]

/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/48.jpg: {   "primary_intent": "information_sharing",   "secondary_intent": null,   "confidence": 0.98,   "reasoning": "The image clearly functions as an event announcement poster, displaying team names (BRA vs SRB), match date, time, and group information. It is structured like official sports event graphics, prioritizing factual dissemination over personal branding or entertainment.",   "visual_evidence": [     "Text stating 'GROUP G THURSDAY 24 NOV KICK-OFF 19:00 GMT'",     "National flags for BRA and SRB",     "Player names (FRED, ANTONY, CASEMIRO) displayed above them",     "Large 'BRA' and 'SRB' text indicating opposing teams",     "Overall layout resembling a sports event poster"   ] } (conf=0.8541)
/home/labad/minxing/code/Qwen3-VL/data_influencer22/sample_30/8.jpg: {   "primary_intent": "information_sharing",   "secondary_intent": null,   "confidence": 0.98,   "reasoning": "The image prominently displays logos of two